[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_02_ridge_lasso.ipynb)

# S07 · 02 — Regularización: Ridge y Lasso

Sprint 7 · M3 (UD 3.3)

**Contexto TurisData.** TurisData abre un **alojamiento nuevo** y solo tiene 120 reservas para estimar su tarifa. Con tan pocos datos y muchas variables (y sus combinaciones), un modelo lineal "normal" se **sobreajusta**: memoriza el ruido. La solución: **regularizar**, es decir, penalizar los coeficientes grandes para que el modelo se mantenga sencillo.

**Este cuaderno tiene 3 ejercicios y un mini-reto.** Los ejercicios 1, 2 y 3 son los mismos que los del cuaderno «en clase» (allí, del 1 al 3).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.base import clone

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
df["temporada_alta"] = df["mes_llegada"].isin([1, 2, 3, 7, 8, 12]).astype(int)
df["pais_origen"] = df["pais_origen"].fillna("desconocido")

columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "peticiones_especiales",
                "cliente_repetidor", "cancelaciones_previas", "mes_llegada", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["precio_noche"]

# Simulamos el alojamiento nuevo: solo 120 reservas para entrenar, el resto para evaluar
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=120, random_state=SEMILLA)
print("Reservas para entrenar:", len(X_train), "· para evaluar:", len(X_test))

prep = ColumnTransformer([("num", StandardScaler(), columnas_num),
                          ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_cat)])
print("Listo ✔")

### 👀 A2 · EJEMPLO — Un modelo generoso con 120 reservas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas columnas tiene el modelo y cuántas filas de entrenamiento hay?
2. ¿Qué R² sale en entrenamiento y en prueba?
3. ¿Por qué el R² de entrenamiento es exactamente 1,000?

In [ ]:
def modelo_con(regresor):
    """Preprocesado + variables de grado 2 + escalado + regresor."""
    return make_pipeline(clone(prep), PolynomialFeatures(2, include_bias=False), StandardScaler(), regresor)

ols = modelo_con(LinearRegression()).fit(X_train, y_train)
print("Número de columnas del modelo:", ols[-1].coef_.shape[0])
print(f"R² en entrenamiento: {ols.score(X_train, y_train):.3f} · R² en prueba: {ols.score(X_test, y_test):.3f}")

### ✏️ A3 · EJERCICIO 1 de 3 — Medir la brecha de sobreajuste

**Tu turno:** Guarda en `r2_train_ols` y `r2_test_ols` los R² de `ols` en train y test, y en `brecha_ols` su diferencia (train − test). ¿Es una brecha sana?

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Cuánto valen `r2_train_ols`, `r2_test_ols` y `brecha_ols`?
2. ¿Es una brecha sana?
3. ¿Qué dos remedios se te ocurren?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: ols.score(X_train, y_train) y ols.score(X_test, y_test)
r2_train_ols = ...
r2_test_ols = ...
brecha_ols = ...

assert r2_train_ols > 0.99, "En train el R² debería ser casi 1: el modelo memoriza"
assert brecha_ols > 0.2, "La brecha debería ser grande (sobreajuste)"
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Trayectorias de los coeficientes: Ridge frente a Lasso

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué les pasa a los coeficientes de Ridge al subir `alpha`?
2. ¿Y a los de Lasso?
3. ¿Por qué el eje horizontal es logarítmico?

In [ ]:
Xp_train = prep.fit_transform(X_train)
nombres = prep.get_feature_names_out()
alphas = np.logspace(-2, 3, 30)
trayectoria_ridge = np.array([Ridge(alpha=a).fit(Xp_train, y_train).coef_ for a in alphas])
trayectoria_lasso = np.array([Lasso(alpha=a, max_iter=50000).fit(Xp_train, y_train).coef_ for a in alphas])

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, tray, titulo in [(ejes[0], trayectoria_ridge, "Ridge (L2): los coeficientes se encogen"),
                         (ejes[1], trayectoria_lasso, "Lasso (L1): los coeficientes llegan a cero")]:
    ax.plot(alphas, tray, lw=1)
    ax.set_xscale("log")
    ax.set(title=titulo, xlabel="alpha (escala logarítmica)")
    ax.axhline(0, color="black", lw=0.8)
ejes[0].set_ylabel("Valor del coeficiente (€)")
plt.tight_layout()
plt.show()

### ✏️ A5 · EJERCICIO 2 de 3 — Ridge con alpha elegido por validación cruzada

**Tu turno:** Crea `ridge` con `modelo_con(RidgeCV(alphas=np.logspace(-1, 4, 30)))` y ajústalo con train. Guarda en `alpha_ridge` el alpha elegido (`ridge[-1].alpha_`) y en `r2_test_ridge` su R² en test.

**Debe salir:** `Ridge: alpha = 25.9 · R² test = 0.728` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Qué `alpha` elige RidgeCV y qué R² da en test?
2. ¿Cuánto mejora frente al modelo sin regularizar (0,716)?
3. ¿Por qué `alpha` se elige con validación cruzada dentro de train y no mirando el test?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: modelo_con(RidgeCV(...)).fit(X_train, y_train); el alpha elegido está en ridge[-1].alpha_
ridge = ...
alpha_ridge = ...
r2_test_ridge = ...

assert alpha_ridge > 0.1, "RidgeCV debe haber elegido un alpha > 0,1"
assert r2_test_ridge > r2_test_ols, "Ridge debería mejorar al modelo sin regularizar en test"
print(f"Ridge: alpha = {alpha_ridge:.1f} · R² test = {r2_test_ridge:.3f}")
print("✔ Ejercicio 2 correcto")

### ✏️ A6 · EJERCICIO 3 de 3 — Lasso: seleccionar variables

**Tu turno:** Crea `lasso` con `modelo_con(LassoCV(cv=5, max_iter=50000, random_state=SEMILLA))`, ajústalo con train y guarda en `r2_test_lasso` su R² en test y en `n_no_cero` cuántos coeficientes son distintos de cero (`np.sum(lasso[-1].coef_ != 0)`).

**Debe salir:** `Lasso: R² test = 0.887 · coeficientes distintos de cero: 27 de 665` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Qué R² en test da Lasso y con cuántos coeficientes se queda?
2. ¿Por qué Lasso mejora tanto más que Ridge (0,887 frente a 0,728)?
3. ¿Qué porcentaje de las columnas conserva?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: el LassoCV es el último paso del pipeline: lasso[-1].coef_
lasso = ...
r2_test_lasso = ...
n_no_cero = ...

assert r2_test_lasso > 0.8, "Lasso debería superar 0,8 de R² en test"
assert n_no_cero < 80, "Lasso debería quedarse con muy pocos coeficientes de los cientos posibles"
print(f"Lasso: R² test = {r2_test_lasso:.3f} · coeficientes distintos de cero: {n_no_cero} de {lasso[-1].coef_.shape[0]}")
print("✔ Ejercicio 3 correcto")

### 👀 A7 · EJEMPLO — Qué variables ha conservado Lasso

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Cuáles son las tres variables con más peso?
2. ¿Qué indica que sean justo esas?
3. Hay combinaciones con peso pequeño (por ejemplo, `cliente_repetidor × pais_origen_NL`, 2,17). ¿Qué cautela hay que tener?

In [ ]:
# ¿Qué variables ha conservado Lasso? (las de mayor peso)
nombres_poly = lasso[:-1].get_feature_names_out()
coef_lasso = pd.Series(lasso[-1].coef_, index=nombres_poly)
principales = coef_lasso[coef_lasso != 0].sort_values(key=np.abs, ascending=False).head(8)
principales.round(2)

### 👀 A8 · EJEMPLO — Tabla comparativa de los tres modelos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué modelo gana en test y por cuánto sobre el modelo sin regularizar?
2. ¿Por qué Lasso tiene el R² de train más bajo (0,957) y aun así es el mejor?
3. ¿En qué caso preferirías Ridge?

In [ ]:
comparativa = pd.DataFrame({
    "R² train": [ols.score(X_train, y_train), ridge.score(X_train, y_train), lasso.score(X_train, y_train)],
    "R² test": [r2_test_ols, r2_test_ridge, r2_test_lasso],
}, index=["Sin regularizar", "Ridge", "Lasso"]).round(3)
comparativa

### ✏️ A9 · MINI-RETO — Curva de R² según el número de reservas

**Tu turno:** Repite el experimento con distintos tamaños de entrenamiento (`tamanos = [60, 120, 300, 1000]`) y guarda en la tabla `curva` el R² en test de la regresión sin regularizar y de Lasso (columnas `"ols"` y `"lasso"`, una fila por tamaño). Usa siempre las mismas 1000 reservas de evaluación (`X_eval`, `y_eval`) para comparar de forma justa.

**Debe salir:** ✔ Mini-reto correcto (y la tabla `curva`)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Cuánto vale la ventaja de Lasso con 60 reservas y con 1000?
2. ¿Qué le pasa al modelo sin regularizar con 300 reservas (R² = −1,06)?
3. ¿Qué pasa con Lasso al pasar de 120 a 300 reservas?

In [ ]:
tamanos = [60, 120, 300, 1000]
X_res, X_eval, y_res, y_eval = train_test_split(X, y, test_size=1000, random_state=SEMILLA)
# pista: para cada tamaño toma X_res.iloc[:n], ajusta los dos modelos y puntúa con X_eval, y_eval; luego pd.DataFrame(...).T.round(3)
curva = ...

assert list(curva.index) == tamanos and list(curva.columns) == ["ols", "lasso"], "Índice = tamaños; columnas 'ols' y 'lasso'"
assert curva.loc[60, "lasso"] > curva.loc[60, "ols"], "Con pocos datos, Lasso debería superar al modelo sin regularizar"
display(curva)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, la regresión lineal, Ridge, Lasso (y sus versiones con validación cruzada), el preprocesado y `clone`.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `reservas_turisdata.csv`, añade `mes_llegada` y `temporada_alta`, rellena los países que faltan con «desconocido» y prepara `X` y el objetivo `y = precio_noche`.
    - **Simula el alojamiento nuevo**: solo 120 reservas para entrenar (`train_size=120`) y el resto (5880) para evaluar.
    - `prep` es el preprocesado: escala las columnas numéricas y convierte las categóricas en columnas 0/1.
- **Qué debe salir:**
    - `Reservas para entrenar: 120 · para evaluar: 5880` y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. El problema: muchas variables, pocos datos

#### A2 · Ejemplo — Un modelo generoso con 120 reservas

Construimos un modelo «generoso»: las variables originales **y todas sus combinaciones de grado 2** (productos y cuadrados). Salen cientos de columnas para solo 120 filas. Solo unas pocas variables influyen de verdad en el precio (tipo de habitación y temporada).

- **Qué hace:**
    - `modelo_con(regresor)` es una función que monta un *pipeline*: preprocesado (`clone(prep)`) → variables de grado 2 (`PolynomialFeatures(2)`) → escalado → el regresor que le pasemos.
    - `ols` es ese pipeline con una `LinearRegression` normal (sin regularizar), ajustado con las 120 reservas.
    - `ols[-1].coef_.shape[0]` cuenta los coeficientes del último paso, es decir, las columnas del modelo; `ols.score(...)` da el R² en train y en test.
- **Qué debe salir:**
    - `Número de columnas del modelo: 665`.
    - `R² en entrenamiento: 1.000 · R² en prueba: 0.716`.
- **Si falla:**
    - `NameError: name 'prep' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas columnas tiene el modelo y cuántas filas de entrenamiento hay?* → **665 columnas** para solo **120 filas**.
2. *¿Qué R² sale en entrenamiento y en prueba?* → **1,000** en entrenamiento y **0,716** en prueba.
3. *¿Por qué el R² de entrenamiento es exactamente 1,000?* → Porque hay **muchas más columnas (665) que filas (120)**: el modelo tiene libertad para pasar por todos los puntos y se aprende de memoria hasta el ruido. Eso es sobreajuste.

</details>

#### A3 · Ejercicio 1 — Medir la brecha de sobreajuste

- **Qué hace:**
    - `ols.score(X_train, y_train)` y `ols.score(X_test, y_test)` dan el R² del modelo en entrenamiento y en prueba.
    - `brecha_ols` es su diferencia (train − test): cuánto peor funciona con datos nuevos.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`. `r2_train_ols` vale 1,000, `r2_test_ols` 0,716 y `brecha_ols` ≈ 0,284.
- **Si falla:**
    - `En train el R² debería ser casi 1` → has puntuado con `X_test`, o con otra tabla.
    - `La brecha debería ser grande (sobreajuste)` → `brecha_ols` está al revés (test − train sale negativa) o no resta los R² correctos.
    - `NameError: name 'ols' is not defined` → falta ejecutar el ejemplo del modelo generoso.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `r2_train_ols`, `r2_test_ols` y `brecha_ols`?* → **1,000**, **0,716** y **≈ 0,284**.
2. *¿Es una brecha sana?* → **No.** Un modelo bien ajustado rinde parecido en train y en test. Una brecha de 0,28 puntos de R² es **sobreajuste**: memoriza en lugar de aprender el patrón.
3. *¿Qué dos remedios se te ocurren?* → **Penalizar los coeficientes** (regularizar: Ridge y Lasso, este cuaderno) o **conseguir más datos** (lo veremos en el mini-reto).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
r2_train_ols = ols.score(X_train, y_train)
r2_test_ols = ols.score(X_test, y_test)
brecha_ols = r2_train_ols - r2_test_ols
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Ridge (L2): encoger los coeficientes

#### A4 · Ejemplo — Trayectorias de los coeficientes: Ridge frente a Lasso

**Idea.** Ridge minimiza el error **más** una penalización `alpha · Σ coeficiente²`.
- `alpha` pequeño → casi una regresión normal.
- `alpha` grande → todos los coeficientes se encogen hacia 0 (modelo más simple, más sesgo).

Ridge **reparte** el peso entre variables parecidas; rara vez deja un coeficiente exactamente en 0. **Es imprescindible escalar** las variables antes, para que la penalización sea justa. Vemos el efecto sobre un modelo sencillo (sin polinomios) al variar `alpha`.

- **Qué hace:**
    - `prep.fit_transform(X_train)` prepara los datos (escalar y one-hot) y `alphas = np.logspace(-2, 3, 30)` crea 30 valores de `alpha` entre 0,01 y 1000.
    - Para cada `alpha`, ajusta un `Ridge` y un `Lasso` con train y guarda sus coeficientes (`trayectoria_ridge`, `trayectoria_lasso`).
    - Dibuja una línea por coeficiente: a la izquierda con Ridge y a la derecha con Lasso. El eje horizontal es `alpha` en escala logarítmica.
- **Qué debe salir:**
    - Dos gráficos con muchas líneas: en Ridge se acercan suavemente a 0 al subir `alpha`; en Lasso van cayendo a 0 una tras otra hasta que todas valen 0.
- **Si falla:**
    - `NameError: name 'prep' is not defined` → falta ejecutar la celda de preparación.
    - `ConvergenceWarning` con Lasso → es un aviso, no un error: el gráfico sale igual.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué les pasa a los coeficientes de Ridge al subir `alpha`?* → Se **encogen poco a poco** hacia 0, pero casi ninguno llega a valer exactamente 0.
2. *¿Y a los de Lasso?* → Se van **anulando uno tras otro** (se pegan al eje en 0) hasta que, con un `alpha` grande, quedan todos a 0: el modelo se queda sin variables.
3. *¿Por qué el eje horizontal es logarítmico?* → Porque `alpha` recorre **cinco órdenes de magnitud** (de 0,01 a 1000): en escala normal, todo lo interesante quedaría apelotonado junto al 0.

</details>

## 3. Elegir `alpha` con validación cruzada

#### A5 · Ejercicio 2 — Ridge con alpha elegido por validación cruzada

`RidgeCV` y `LassoCV` prueban una lista de valores de `alpha` con validación cruzada **dentro de train** y se quedan con el mejor. El test **no se toca** hasta el final.

- **Qué hace:**
    - `modelo_con(RidgeCV(alphas=np.logspace(-1, 4, 30)))` monta el pipeline con un Ridge que prueba 30 valores de `alpha` entre 0,1 y 10 000.
    - `.fit(X_train, y_train)` lo ajusta con train, eligiendo `alpha` por validación cruzada.
    - `ridge[-1]` es el último paso del pipeline (el Ridge): su `.alpha_` es el valor elegido. `ridge.score(X_test, y_test)` da el R² en test.
- **Qué debe salir:**
    - `Ridge: alpha = 25.9 · R² test = 0.728`.
    - `✔ Ejercicio 2 correcto`.
- **Si falla:**
    - `RidgeCV debe haber elegido un alpha > 0,1` → has sacado otro valor que `ridge[-1].alpha_`.
    - `Ridge debería mejorar al modelo sin regularizar en test` → has puntuado con train o no has ajustado `ridge` con `X_train, y_train`.
    - `NameError: name 'modelo_con' is not defined` → falta ejecutar el ejemplo del modelo generoso.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué `alpha` elige RidgeCV y qué R² da en test?* → **`alpha` ≈ 25,9** y **R² = 0,728** en test.
2. *¿Cuánto mejora frente al modelo sin regularizar (0,716)?* → Solo **0,012**: Ridge reduce el sobreajuste pero **no lo resuelve**, porque conserva las 665 variables (las encoge, no las quita).
3. *¿Por qué `alpha` se elige con validación cruzada dentro de train y no mirando el test?* → Porque si eligiéramos `alpha` mirando test, el modelo estaría ajustado al examen y el R² de test dejaría de ser una estimación honesta.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
ridge = modelo_con(RidgeCV(alphas=np.logspace(-1, 4, 30))).fit(X_train, y_train)
alpha_ridge = ridge[-1].alpha_
r2_test_ridge = ridge.score(X_test, y_test)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Lasso (L1): seleccionar variables

#### A6 · Ejercicio 3 — Lasso: seleccionar variables

**Idea.** Lasso penaliza `alpha · Σ |coeficiente|`. Esta penalización tiene un efecto curioso: **lleva a cero** los coeficientes de las variables poco útiles. El resultado es un modelo **disperso** (*sparse*) que además hace **selección de variables**. Es ideal cuando sospechas que solo unas pocas variables importan.

- **Qué hace:**
    - `modelo_con(LassoCV(cv=5, max_iter=50000, random_state=SEMILLA))` monta el pipeline con un Lasso que elige `alpha` con validación cruzada de 5 folds.
    - `r2_test_lasso` es su R² en test; `lasso[-1].coef_ != 0` marca los coeficientes que no son cero y `np.sum(...)` los cuenta (`int(...)` lo deja como entero).
- **Qué debe salir:**
    - `Lasso: R² test = 0.887 · coeficientes distintos de cero: 27 de 665`.
    - `✔ Ejercicio 3 correcto`.
- **Si falla:**
    - `Lasso debería superar 0,8 de R² en test` → no has ajustado `lasso` con `X_train, y_train` o has puntuado con train.
    - `Lasso debería quedarse con muy pocos coeficientes` → `n_no_cero` cuenta los coeficientes *iguales* a cero, no los distintos (revisa `!= 0`).
    - `NameError: name 'modelo_con' is not defined` → falta ejecutar el ejemplo del modelo generoso.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué R² en test da Lasso y con cuántos coeficientes se queda?* → **R² = 0,887** y se queda con **27 de los 665** coeficientes.
2. *¿Por qué Lasso mejora tanto más que Ridge (0,887 frente a 0,728)?* → Porque en estos datos **pocas variables lo explican casi todo** (tipo de habitación y temporada): Lasso descarta el resto (638 columnas de ruido), mientras que Ridge las conserva todas, aunque encogidas.
3. *¿Qué porcentaje de las columnas conserva?* → 27 de 665: **≈ 4 %**. Es un modelo mucho más simple y fácil de explicar.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
lasso = modelo_con(LassoCV(cv=5, max_iter=50000, random_state=SEMILLA)).fit(X_train, y_train)
r2_test_lasso = lasso.score(X_test, y_test)
n_no_cero = int(np.sum(lasso[-1].coef_ != 0))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A7 · Ejemplo — Qué variables ha conservado Lasso

- **Qué hace:**
    - `lasso[:-1].get_feature_names_out()` da el nombre de cada columna del modelo (las originales y sus combinaciones); `coef_lasso` los une con sus coeficientes.
    - `coef_lasso[coef_lasso != 0]` se queda con los no nulos; `.sort_values(key=np.abs, ascending=False)` los ordena por tamaño, sin importar el signo; `.head(8)` muestra los 8 primeros.
- **Qué debe salir:**
    - Una lista de 8 filas. Las tres primeras: `cat__tipo_habitacion_suite` 16,16, `cat__tipo_habitacion_estandar` −14,03 y `num__temporada_alta` 7,18.
- **Cómo leerlo:**
    - Los coeficientes están sobre variables **escaladas**: miden € por cada desviación típica de la variable, y por eso se pueden comparar entre sí.
    - Un nombre con dos variables (`num__ninos cat__tipo_habitacion_suite`) es una **combinación** (producto) de ambas.
- **Si falla:**
    - `NameError: name 'lasso' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuáles son las tres variables con más peso?* → **Suite** (+16,16), **estándar** (−14,03) y **temporada alta** (+7,18).
2. *¿Qué indica que sean justo esas?* → Que Lasso ha **encontrado la señal real** entre 665 columnas: el tipo de habitación y la temporada son lo que de verdad determina el precio.
3. *Hay combinaciones con peso pequeño (por ejemplo, `cliente_repetidor × pais_origen_NL`, 2,17). ¿Qué cautela hay que tener?* → Con solo 120 reservas, los pesos pequeños son **menos fiables** (podrían ser casualidades): se interpretan con cautela, y el mensaje principal viene de las variables grandes.

</details>

## 5. Comparativa final

#### A8 · Ejemplo — Tabla comparativa de los tres modelos

**Idea.** Ninguno gana siempre:
- **Ridge** es mejor cuando **muchas** variables aportan un poco.
- **Lasso** es mejor cuando **pocas** variables lo explican casi todo (nuestro caso) y además da un modelo fácil de explicar.
- **ElasticNet** mezcla ambas penalizaciones (sirve cuando hay variables muy correlacionadas).

- **Qué hace:**
    - Monta una tabla con el R² en entrenamiento y en prueba de los tres modelos (sin regularizar, Ridge y Lasso), redondeado a 3 decimales.
- **Qué debe salir:**
    - Una tabla: `Sin regularizar` 1,000 / 0,716 · `Ridge` 0,998 / 0,728 · `Lasso` 0,957 / 0,887 (R² train / R² test).
- **Si falla:**
    - `NameError: name 'ridge' is not defined` o `'lasso'` → faltan los ejercicios 2 y 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué modelo gana en test y por cuánto sobre el modelo sin regularizar?* → **Lasso** (0,887): **0,171 más** que sin regularizar (0,716).
2. *¿Por qué Lasso tiene el R² de train más bajo (0,957) y aun así es el mejor?* → Porque **no memoriza**: la penalización le hace renunciar a algo de ajuste en train, y por eso generaliza mejor en datos nuevos.
3. *¿En qué caso preferirías Ridge?* → Cuando **muchas variables aportan un poco** (el efecto se reparte entre ellas) o hay variables muy parecidas entre sí: Ridge las mantiene todas, encogidas.

</details>

## 6. Mini-reto: ¿cuántos datos hacen falta?

#### A9 · Mini-reto — Curva de R² según el número de reservas

- **Qué hace:**
    - `train_test_split(X, y, test_size=1000)` aparta 1000 reservas fijas para evaluar (`X_eval`, `y_eval`) y deja el resto (`X_res`, `y_res`) para entrenar.
    - Para cada tamaño `n` de `tamanos`, entrena con las primeras `n` reservas dos modelos: la regresión sin regularizar y Lasso, ambos con `modelo_con(...)`.
    - Puntúa los dos siempre con las **mismas** 1000 reservas de evaluación y guarda los R² en un diccionario; `pd.DataFrame(filas).T.round(3)` lo convierte en la tabla `curva` (una fila por tamaño).
- **Qué debe salir:**
    - `✔ Mini-reto correcto` y la tabla `curva`: `60` ols 0,449 / lasso 0,691 · `120` 0,704 / 0,848 · `300` −1,062 / 0,913 · `1000` 0,836 / 0,925.
- **Si falla:**
    - `Índice = tamaños; columnas 'ols' y 'lasso'` → falta `.T` (el DataFrame sale con los tamaños en las columnas) o las columnas tienen otro nombre.
    - `Con pocos datos, Lasso debería superar al modelo sin regularizar` → has puntuado con `X_res` en vez de con `X_eval, y_eval`, o has usado todas las filas en cada tamaño.
    - `KeyError`/`IndexError` → el filtro `X_res.iloc[:n]` y `y_res.iloc[:n]` debe usar el mismo `n`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale la ventaja de Lasso con 60 reservas y con 1000?* → Con 60: **0,691 frente a 0,449** (0,24 más). Con 1000: **0,925 frente a 0,836** (0,09 más). La ventaja se reduce.
2. *¿Qué le pasa al modelo sin regularizar con 300 reservas (R² = −1,06)?* → Que es **peor que predecir la media**: con 300 filas y 665 columnas está justo en la zona donde el modelo sin regularizar es más inestable y sus predicciones se disparan. Lasso, en cambio, da 0,913.
3. *¿Qué pasa con Lasso al pasar de 120 a 300 reservas?* → Mejora de **0,848 a 0,913**: con más datos todos los modelos mejoran, pero Lasso necesita muchos menos para rendir bien.

</details>

## 7. Para reflexionar

1. ¿Por qué con 1000 reservas la ventaja de regularizar se reduce?
2. Ridge no dejó ningún coeficiente en cero y Lasso dejó solo unas pocas decenas de los 665 (mira el recuento de arriba). ¿Qué preferirías si el cliente pide "las 5 cosas que más influyen en el precio"? ¿Cómo sacarías esas 5 del modelo?
3. ¿Qué habría pasado si no hubiéramos escalado las variables?
4. ¿Por qué `alpha` se elige con validación cruzada en train y no mirando el test?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque con 1000 reservas hay muchos más datos que variables útiles y el sobreajuste baja por sí solo: el modelo sin regularizar ya generaliza (R² 0,836) y la ventaja de Lasso (0,925) pasa de 0,24 puntos con 60 reservas a 0,09.
2. **Lasso**: al llevar a cero casi todo, deja una lista corta. Se ordenan los coeficientes no nulos por valor absoluto (con las variables escaladas son comparables) y se toman los 5 primeros; aquí saldrían el tipo de habitación y la temporada. Ridge conserva todas las variables, y esa lista corta no la da.
3. La penalización trataría de forma injusta a las variables: una columna con valores pequeños necesita un coeficiente grande y se castigaría más, y otra con valores grandes, menos. El resultado dependería de las unidades (euros, días…) y no de la importancia real.
4. Porque si eligiéramos `alpha` mirando test, estaríamos ajustando el modelo al examen: el R² de test dejaría de ser una estimación honesta. Con validación cruzada dentro de train, el test queda intacto hasta el final.

</details>

## 8. Resumen: qué has aprendido
- Muchas variables + pocos datos = sobreajuste (train perfecto, test mediocre).
- Ridge encoge coeficientes; Lasso además los anula (selección de variables).
- `alpha` se elige con `RidgeCV`/`LassoCV`; siempre con las variables escaladas.
- Con más datos, la regularización pesa menos.